Data Cleaning in Pandas

In [83]:
import pandas as pd

In [121]:
#understand the dataset
df_game = pd.read_csv(r"C:\Users\zacle\Desktop\Serene\Project\ETL\extract\raw_data\game.csv")
print(df_game.dtypes)

game_id                    int64
season                     int64
type                      object
date_time_GMT             object
away_team_id               int64
home_team_id               int64
away_goals                 int64
home_goals                 int64
outcome                   object
home_rink_side_start      object
venue                     object
venue_link                object
venue_time_zone_id        object
venue_time_zone_offset     int64
venue_time_zone_tz        object
dtype: object


Create New Dataframe

In [124]:
# Create a new dataframe from df_game_plays (copying original to preserve data)
df_clean_game = df_game.copy()

Check : Null 

In [127]:
# check for missing or null values
df_game.isnull().sum()

game_id                      0
season                       0
type                         0
date_time_GMT                0
away_team_id                 0
home_team_id                 0
away_goals                   0
home_goals                   0
outcome                      0
home_rink_side_start      1196
venue                        0
venue_link                   0
venue_time_zone_id           0
venue_time_zone_offset       0
venue_time_zone_tz           0
dtype: int64

Check : Duplicates

In [130]:
# Count unique 'game_id' values
unique_ids = df_clean_game['game_id'].nunique()

# Count total number of rows
total_rows = len(df_clean_game)

# Display the results
print(f"Unique game_ids: {unique_ids}")
print(f"Total rows: {total_rows}")
print(f"There are {total_rows - unique_ids} rows with duplicates.")


Unique game_ids: 23735
Total rows: 26305
There are 2570 rows with duplicates.


In [132]:
# Check duplicates for game_id, assigned to duplicates_game_id
duplicates_game_id = df_clean_game[df_clean_game.duplicated(subset=['game_id'], keep=False)]

# Sort rows with duplicated results in ascending order
duplicates_game_id_sorted = duplicates_game_id.sort_values(by='game_id', ascending=True)

# Print duplicate rows
duplicates_game_id_sorted

# Upon visual inspection, there are 4946 rows where the same game_id containing identical 
# results in the game_id column but different values under the 'away_team_id' and 'home_team_id'

,game_id,season,type,date_time_GMT,away_team_id,home_team_id,away_goals,home_goals,outcome,home_rink_side_start,venue,venue_link,venue_time_zone_id,venue_time_zone_offset,venue_time_zone_tz
23585,2018020001,20182019,R,2018-10-03T23:00:00Z,8,10,2,3,home win OT,right,Scotiabank Arena,/api/v1/venues/null,America/Toronto,-4,EDT
23589,2018020001,20182019,R,2018-10-03T23:00:00Z,8,10,2,3,home win OT,right,Scotiabank Arena,/api/v1/venues/null,America/Toronto,-4,EDT
23586,2018020002,20182019,R,2018-10-03T23:30:00Z,6,15,0,7,home win REG,right,Capital One Arena,/api/v1/venues/5094,America/New_York,-5,EST
23590,2018020002,20182019,R,2018-10-03T23:30:00Z,6,15,0,7,home win REG,right,Capital One Arena,/api/v1/venues/5094,America/New_York,-5,EST
23587,2018020003,20182019,R,2018-10-04T02:00:00Z,20,23,2,5,home win REG,right,Rogers Arena,/api/v1/venues/5073,America/Vancouver,-7,PDT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22694,2019040651,20192020,A,2020-01-26T01:15:00Z,88,87,5,9,home win tbc,NaN,Enterprise Center,/api/v1/venues/5076,America/New_York,-5,EST
22695,2019040652,20192020,A,2020-01-26T02:15:00Z,90,89,10,5,away win tbc,NaN,Enterprise Center,/api/v1/venues/5076,America/New_York,-5,EST
22698,2019040652,20192020,A,2020-01-26T02:15:00Z,90,89,10,5,away win tbc,NaN,Enterprise Center,/api/v1/venues/5076,America/New_York,-5,EST
22699,2019040653,20192020,A,2020-01-26T03:15:00Z,87,90,4,5,home win tbc,NaN,Enterprise Center,/api/v1/venues/5076,America/New_York,-5,EST


In [134]:
# Additional 2nd Duplicate Check : 'game_id' alone might not be sufficient to determine duplicates 
# A single game might be recorded multiple times for different teams, e.g., home vs. away based on columns 'game_id', 'away_team_id', and 'home_team_id'
duplicates_game_id = df_clean_game[df_clean_game.duplicated(subset=['game_id', 'away_team_id', 'home_team_id'], keep=False)]

# Sort rows with duplicated results in ascending order by 'game_id', 'away_team_id', and 'home_team_id'
duplicates_game_id_sorted = duplicates_game_id.sort_values(by=['game_id', 'away_team_id', 'home_team_id'], ascending=True)

# Print the duplicated rows
duplicates_game_id_sorted

# Upon visual inspection, there are 5140 rows where the same game_id, away_team_id, home_team_id combo containing true duplicates (identical results)

,game_id,season,type,date_time_GMT,away_team_id,home_team_id,away_goals,home_goals,outcome,home_rink_side_start,venue,venue_link,venue_time_zone_id,venue_time_zone_offset,venue_time_zone_tz
23585,2018020001,20182019,R,2018-10-03T23:00:00Z,8,10,2,3,home win OT,right,Scotiabank Arena,/api/v1/venues/null,America/Toronto,-4,EDT
23589,2018020001,20182019,R,2018-10-03T23:00:00Z,8,10,2,3,home win OT,right,Scotiabank Arena,/api/v1/venues/null,America/Toronto,-4,EDT
23586,2018020002,20182019,R,2018-10-03T23:30:00Z,6,15,0,7,home win REG,right,Capital One Arena,/api/v1/venues/5094,America/New_York,-5,EST
23590,2018020002,20182019,R,2018-10-03T23:30:00Z,6,15,0,7,home win REG,right,Capital One Arena,/api/v1/venues/5094,America/New_York,-5,EST
23587,2018020003,20182019,R,2018-10-04T02:00:00Z,20,23,2,5,home win REG,right,Rogers Arena,/api/v1/venues/5073,America/Vancouver,-7,PDT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22697,2019040651,20192020,A,2020-01-26T01:15:00Z,88,87,5,9,home win tbc,NaN,Enterprise Center,/api/v1/venues/5076,America/New_York,-5,EST
22695,2019040652,20192020,A,2020-01-26T02:15:00Z,90,89,10,5,away win tbc,NaN,Enterprise Center,/api/v1/venues/5076,America/New_York,-5,EST
22698,2019040652,20192020,A,2020-01-26T02:15:00Z,90,89,10,5,away win tbc,NaN,Enterprise Center,/api/v1/venues/5076,America/New_York,-5,EST
22696,2019040653,20192020,A,2020-01-26T03:15:00Z,87,90,4,5,home win tbc,NaN,Enterprise Center,/api/v1/venues/5076,America/New_York,-5,EST


In [136]:
# Remove duplicates based on 'game_id', 'away_team_id', and 'home_team_id'
df_clean_game = df_clean_game.drop_duplicates(subset=['game_id', 'away_team_id', 'home_team_id'], keep='first')

# Display the cleaned dataframe (first 5 rows as an example)
df_clean_game

,game_id,season,type,date_time_GMT,away_team_id,home_team_id,away_goals,home_goals,outcome,home_rink_side_start,venue,venue_link,venue_time_zone_id,venue_time_zone_offset,venue_time_zone_tz
0,2016020045,20162017,R,2016-10-19T00:30:00Z,4,16,4,7,home win REG,right,United Center,/api/v1/venues/null,America/Chicago,-5,CDT
1,2017020812,20172018,R,2018-02-07T00:00:00Z,24,7,4,3,away win OT,left,KeyBank Center,/api/v1/venues/null,America/New_York,-4,EDT
2,2015020314,20152016,R,2015-11-24T01:00:00Z,21,52,4,1,away win REG,right,MTS Centre,/api/v1/venues/null,America/Winnipeg,-5,CDT
3,2015020849,20152016,R,2016-02-17T00:00:00Z,52,12,1,2,home win REG,right,PNC Arena,/api/v1/venues/null,America/New_York,-4,EDT
4,2017020586,20172018,R,2017-12-30T03:00:00Z,20,24,1,2,home win REG,left,Honda Center,/api/v1/venues/null,America/Los_Angeles,-7,PDT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26295,2018030413,20182019,P,2019-06-02T00:00:00Z,6,19,7,2,away win REG,left,Enterprise Center,/api/v1/venues/5076,America/Chicago,-5,CDT
26297,2018030414,20182019,P,2019-06-04T00:00:00Z,6,19,2,4,home win REG,left,Enterprise Center,/api/v1/venues/5076,America/Chicago,-6,CST
26299,2018030415,20182019,P,2019-06-07T00:00:00Z,19,6,2,1,away win REG,left,TD Garden,/api/v1/venues/5085,America/New_York,-5,EST
26301,2018030416,20182019,P,2019-06-10T00:00:00Z,6,19,5,1,away win REG,left,Enterprise Center,/api/v1/venues/5076,America/Chicago,-6,CST


In [138]:
# Final Duplicate Check : On columns 'game_id', 'away_team_id', and 'home_team_id'
duplicates_game_id = df_clean_game[df_clean_game.duplicated(subset=['game_id', 'away_team_id', 'home_team_id'], keep=False)]

# Print the duplicated rows
duplicates_game_id_sorted

# Results : No Duplicates. Resolved!

,game_id,season,type,date_time_GMT,away_team_id,home_team_id,away_goals,home_goals,outcome,home_rink_side_start,venue,venue_link,venue_time_zone_id,venue_time_zone_offset,venue_time_zone_tz
23585,2018020001,20182019,R,2018-10-03T23:00:00Z,8,10,2,3,home win OT,right,Scotiabank Arena,/api/v1/venues/null,America/Toronto,-4,EDT
23589,2018020001,20182019,R,2018-10-03T23:00:00Z,8,10,2,3,home win OT,right,Scotiabank Arena,/api/v1/venues/null,America/Toronto,-4,EDT
23586,2018020002,20182019,R,2018-10-03T23:30:00Z,6,15,0,7,home win REG,right,Capital One Arena,/api/v1/venues/5094,America/New_York,-5,EST
23590,2018020002,20182019,R,2018-10-03T23:30:00Z,6,15,0,7,home win REG,right,Capital One Arena,/api/v1/venues/5094,America/New_York,-5,EST
23587,2018020003,20182019,R,2018-10-04T02:00:00Z,20,23,2,5,home win REG,right,Rogers Arena,/api/v1/venues/5073,America/Vancouver,-7,PDT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22697,2019040651,20192020,A,2020-01-26T01:15:00Z,88,87,5,9,home win tbc,NaN,Enterprise Center,/api/v1/venues/5076,America/New_York,-5,EST
22695,2019040652,20192020,A,2020-01-26T02:15:00Z,90,89,10,5,away win tbc,NaN,Enterprise Center,/api/v1/venues/5076,America/New_York,-5,EST
22698,2019040652,20192020,A,2020-01-26T02:15:00Z,90,89,10,5,away win tbc,NaN,Enterprise Center,/api/v1/venues/5076,America/New_York,-5,EST
22696,2019040653,20192020,A,2020-01-26T03:15:00Z,87,90,4,5,home win tbc,NaN,Enterprise Center,/api/v1/venues/5076,America/New_York,-5,EST


Consistent Column Name - Python PEP Style Guide

In [141]:
#Rename columns for consistency
#Syntax - Rename specific column : df = df.rename(columns={'old_column_name': 'new_column_name'})
#Syntax - Replace space with underscore: df.columns = df.columns.str.replace(' ', '_').str.lower()
#Syntax - Convert column name to lowercase: df.columns = df.columns.str.lower()

df_clean_game = df_clean_game.rename(columns={'type': 'game_type'}) #Convert column name 'type' to 'game_type'. type is a built-in function.

In [143]:
df_clean_game.columns = df_clean_game.columns.str.lower()  # Converts date_time_GMT and all other column names to lowercase

In [145]:
df_clean_game.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23735 entries, 0 to 26303
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   game_id                 23735 non-null  int64 
 1   season                  23735 non-null  int64 
 2   game_type               23735 non-null  object
 3   date_time_gmt           23735 non-null  object
 4   away_team_id            23735 non-null  int64 
 5   home_team_id            23735 non-null  int64 
 6   away_goals              23735 non-null  int64 
 7   home_goals              23735 non-null  int64 
 8   outcome                 23735 non-null  object
 9   home_rink_side_start    22636 non-null  object
 10  venue                   23735 non-null  object
 11  venue_link              23735 non-null  object
 12  venue_time_zone_id      23735 non-null  object
 13  venue_time_zone_offset  23735 non-null  int64 
 14  venue_time_zone_tz      23735 non-null  object
dtypes: int6

Change Data Type

In [148]:
# Change data type syntax - df['column_name'] = df['column_name'].astype('desired_data_type')
df_clean_game['game_id'] = df_clean_game['game_id'].astype(str)
df_clean_game['home_team_id'] = df_clean_game['home_team_id'].astype(str)
df_clean_game['away_team_id'] = df_clean_game['away_team_id'].astype(str)

In [150]:
# Convert data type : 'game_date_time_gmt' to timestamp for consistency
df_clean_game['date_time_gmt'] = pd.to_datetime(df_clean_game['date_time_gmt'])

# Convert 'date_time_gmt' to datetime (timestamp)
df_clean_game['date_time_gmt'] = pd.to_datetime(df_clean_game['date_time_gmt'])

print(df_clean_game.dtypes)

game_id                                object
season                                  int64
game_type                              object
date_time_gmt             datetime64[ns, UTC]
away_team_id                           object
home_team_id                           object
away_goals                              int64
home_goals                              int64
outcome                                object
home_rink_side_start                   object
venue                                  object
venue_link                             object
venue_time_zone_id                     object
venue_time_zone_offset                  int64
venue_time_zone_tz                     object
dtype: object


In [152]:
# Extract the 'outcome' column into two new columns 'hoa' (winner) and 'settled_in'=(REG)ular or OT(overtime)
df_clean_game[['hoa', 'settled_in']] = df_clean_game['outcome'].str.split(' ', expand=True, n=2)[[0, 2]]

# Verify the result
print(df_clean_game[['outcome', 'hoa', 'settled_in']])

            outcome   hoa settled_in
0      home win REG  home        REG
1       away win OT  away         OT
2      away win REG  away        REG
3      home win REG  home        REG
4      home win REG  home        REG
...             ...   ...        ...
26295  away win REG  away        REG
26297  home win REG  home        REG
26299  away win REG  away        REG
26301  away win REG  away        REG
26303  away win REG  away        REG

[23735 rows x 3 columns]


Remove Redundant Columns

In [155]:
df_clean_game.head(5)

,game_id,season,game_type,date_time_gmt,away_team_id,home_team_id,away_goals,home_goals,outcome,home_rink_side_start,venue,venue_link,venue_time_zone_id,venue_time_zone_offset,venue_time_zone_tz,hoa,settled_in
0,2016020045,20162017,R,2016-10-19 00:30:00+00:00,4,16,4,7,home win REG,right,United Center,/api/v1/venues/null,America/Chicago,-5,CDT,home,REG
1,2017020812,20172018,R,2018-02-07 00:00:00+00:00,24,7,4,3,away win OT,left,KeyBank Center,/api/v1/venues/null,America/New_York,-4,EDT,away,OT
2,2015020314,20152016,R,2015-11-24 01:00:00+00:00,21,52,4,1,away win REG,right,MTS Centre,/api/v1/venues/null,America/Winnipeg,-5,CDT,away,REG
3,2015020849,20152016,R,2016-02-17 00:00:00+00:00,52,12,1,2,home win REG,right,PNC Arena,/api/v1/venues/null,America/New_York,-4,EDT,home,REG
4,2017020586,20172018,R,2017-12-30 03:00:00+00:00,20,24,1,2,home win REG,left,Honda Center,/api/v1/venues/null,America/Los_Angeles,-7,PDT,home,REG


In [157]:
df_clean_game.to_csv(r"C:\Users\zacle\Desktop\Serene\Project\ETL\clean\game.csv", index=False)

In [159]:
#Issue : Data type for date_time_gmt and game_id tends to convert back to object and integer. 
# During the loading stage to PostgreSQL, must parse the date and string values.
df = pd.read_csv(r"C:\Users\zacle\Desktop\Serene\Project\ETL\clean\game.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23735 entries, 0 to 23734
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   game_id                 23735 non-null  int64 
 1   season                  23735 non-null  int64 
 2   game_type               23735 non-null  object
 3   date_time_gmt           23735 non-null  object
 4   away_team_id            23735 non-null  int64 
 5   home_team_id            23735 non-null  int64 
 6   away_goals              23735 non-null  int64 
 7   home_goals              23735 non-null  int64 
 8   outcome                 23735 non-null  object
 9   home_rink_side_start    22636 non-null  object
 10  venue                   23735 non-null  object
 11  venue_link              23735 non-null  object
 12  venue_time_zone_id      23735 non-null  object
 13  venue_time_zone_offset  23735 non-null  int64 
 14  venue_time_zone_tz      23735 non-null  object
 15  ho